In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import glob
import math

# Loading Data

In [2]:
testRunFolderName = "TestRun5/AfterCodeEdit"
coordFileType = "Unity"
trialNum = 0

In [3]:
def getFilePath(fileNamePattern: str, trialNumber = 0) -> str:
    folderPath = os.path.join(os.getcwd(), testRunFolderName)
    pattern = os.path.join(folderPath, fileNamePattern)
    # file_path = os.path.join(folderPath, file_name)
    matching_files = glob.glob(pattern)
    matching_files.sort()

    if matching_files:
        file_path = matching_files[trialNumber]
    else:
        raise FileNotFoundError("No file found")
    
    return file_path

In [4]:
csvFileName = coordFileType + "Coordinates_Trial_" + str(trialNum) + ".csv"
csvFilePath = getFilePath(csvFileName)

triggeredEventsFileName = "TriggeredEvents_Trial_" + str(trialNum) + ".csv"
triggeredEventsFilePath = getFilePath(triggeredEventsFileName)

In [5]:
pose_df = pd.read_csv(csvFilePath)

timeStamp_df = pd.read_csv(triggeredEventsFilePath)

# Filtering Data

In [6]:
def get3dPointDistance(pose1, pose2) -> float:
    x = math.pow((pose1[0] - pose2[0]), 2)
    y = math.pow((pose1[1] - pose2[1]), 2)
    z = math.pow((pose1[2] - pose2[2]), 2)
    distance = math.sqrt(x + y + z)
    return distance

In [18]:
def get2DPointDistance(pose1, pose2) -> float:
    x = math.pow((pose1[0] - pose2[0]), 2)
    z = math.pow((pose1[2] - pose2[2]), 2)
    distance = math.sqrt(x + z)
    return distance

### Add waypoints to list

In [7]:
waypointPoses = []
for rowNum in range(10):
    AtWaypoint1 = timeStamp_df["AtWaypoint Time"].loc[timeStamp_df.index[rowNum]]
    LeavingWaypoint1 = timeStamp_df["LeavingWaypoint Time"].loc[timeStamp_df.index[rowNum]]
    mask = (pose_df["Timestamp"] >= AtWaypoint1) & (pose_df["Timestamp"] <= LeavingWaypoint1)

    poseData_df = pose_df[mask]
    # samplesCount = poseData_df.shape[0]
    # print("Valid Times: " + str(samplesCount))

    waypointPoses.append(poseData_df)

### Get Centroids

In [24]:
waypointCentroids = []

for wpData_df in waypointPoses:
    centroid = (wpData_df["X"].mean(), wpData_df["Y"].mean(), wpData_df["Z"].mean())
    print("%.3f , %.3f, %.3f" % (centroid[0], centroid[1], centroid[2]))
    waypointCentroids.append(centroid)


waypointTest_df = pd.DataFrame(waypointCentroids)

13.560 , -0.802, 10.651
15.440 , -0.779, 12.291
17.311 , -0.750, 13.959
19.219 , -0.730, 15.638
21.106 , -0.709, 17.271
22.989 , -0.680, 18.920
24.892 , -0.664, 20.574
26.777 , -0.644, 22.183
28.620 , -0.628, 23.845
30.464 , -0.603, 25.463


## Errors

### Distance First and Last

In [22]:
pathLength = 22.5 # meters
pointDistance = get3dPointDistance(waypointCentroids[0], waypointCentroids[9])
error = pathLength - pointDistance
print("Distance: %.4f m" % pointDistance)
print("Error: %.4f m" % error)

Distance: 22.4760 m
Error: 0.0240 m


### Distance Between Points

In [ ]:
betweenWaypointErrors = []
waypointActualDistance = 2.5 #meters
for index in range(9):
    if index == 10:
        break
    pointDistance = get3dPointDistance(waypointCentroids[index], waypointCentroids[index + 1])
    error = abs(waypointActualDistance - pointDistance)
    betweenWaypointErrors.append(error)

print(["{0:0.4f}m".format(i) for i in betweenWaypointErrors])


['0.0052m', '0.0067m', '0.0415m', '0.0045m', '0.0032m', '0.0214m', '0.0215m', '0.0187m', '0.0462m']


In [19]:
between2DWaypointErrors = []
waypointActualDistance = 2.5 #meters
for index in range(9):
    if index == 10:
        break
    pointDistance = get2DPointDistance(waypointCentroids[index], waypointCentroids[index + 1])
    error = abs(waypointActualDistance - pointDistance)
    between2DWaypointErrors.append(error)

print(["{0:0.4f}m".format(i) for i in between2DWaypointErrors])

['0.0054m', '0.0066m', '0.0414m', '0.0046m', '0.0030m', '0.0213m', '0.0216m', '0.0188m', '0.0463m']
